# What the hell is FP??

Original number

13.25

↓

Binary

1101.01

↓

Normalize

1.10101 × 2³

↓

Sign     = 0
Exponent = 3
Mantissa = 10101

What is a Floating Point Number?

A floating-point number is just a way of storing real numbers (numbers with decimals).

Instead of storing the entire decimal number directly, computers store it in scientific notation.
For example,

12345

can be written as

1.2345 × 10^4

Computers do the same thing, but with base 2 instead of base 10.

Example:

20

Binary:
10100

Scientific binary:

1.01 × 2^4

```python
Value = (-1)^Sign × Mantissa × 2^Exponent
```

Three Components

Suppose we want to store

13.25

Binary representation:

1101.01

Normalize it:

1.10101 × 2^3

Now we have

Sign = 0
Exponent = 3
Mantissa = 10101

These three fields are what every FP format stores.'



Sign Bit

Very easy.

0 = Positive

1 = Negative

Examples

+5

Sign = 0
-5

Sign = 1

Only one bit is needed



Exponent

Exponent decides the range.

Example

1 × 2^0 = 1

1 × 2^1 = 2

1 × 2^2 = 4

1 × 2^3 = 8

1 × 2^4 = 16

The larger the exponent,

the larger numbers you can represent.




Mantissa

Mantissa decides the precision.

Example

Suppose

1.000
=1

Now

1.001
=1.125

Now

1.010
=1.25

+---+--------+-----------------------+
| S | Exp(8) | Mantissa (23)         |
+---+--------+-----------------------+

Example

Store

13.25

Binary

1101.01

Normalize

1.10101 × 2^3

FP32 stores approximately

Sign

0

Exponent

3 (biased internally)

Mantissa

10101000000000000000000

## FP16
FP16 (Half Precision)

Now reduce everything.

Layout

1 Sign

5 Exponent

10 Mantissa

Example

Again

13.25

Still representable.

But now

only

10 mantissa bits.

Instead of

3.1415926535

FP16 stores something like

3.140625

# BF16
BF16 (Brain Floating Point)

Google designed BF16 because they noticed something important:

Neural networks often need large range, but not necessarily high precision.

So BF16 keeps the same exponent size as FP32 while shrinking the mantissa.

Layout:

1 Sign
8 Exponent
7 Mantissa

Diagram:

+---+--------+---------+
| S | Exp(8) | Man(7)  |
+---+--------+---------+

Compared to FP16:

Format	Exponent	Mantissa
FP16	5	10
BF16	8	7
Why is this useful?

Suppose your values are:

0.0000001
10
100000

FP16 may overflow or underflow because its exponent range is limited.

BF16 can represent all of these because it inherits FP32's exponent range.

The trade-off is that the decimal precision is lower.

This is why BF16 has become the default training format on modern accelerators like NVIDIA Hopper and Blackwell, as well as Google's TPUs.

# FP8 FP4
FP8

Now we enter the world of aggressive optimization.

Layout (one common variant, E4M3):

1 Sign
4 Exponent
3 Mantissa

Diagram:

+---+------+------+
| S | Exp  | Man  |
+---+------+------+

Only 8 bits total.

Memory:

100M weights

×

1 byte

=

100 MB

That's a 4× reduction compared to FP32.

What do you lose?

Very fine precision.

Suppose you want to store:

1.23456

FP8 might store:

1.25

Another value:

0.117

could become

0.125

These rounding errors are much larger than in FP16, but many neural network layers tolerate them surprisingly well, especially during inference.

FP4

This is where things become challenging.

You only have 4 bits.

A conceptual layout is:

1 Sign
2 Exponent
1 Mantissa

Diagram:

+---+----+---+
| S |Exp | M |
+---+----+---+

There are only:

2^4 = 16

possible bit patterns.

That means the format can represent only a tiny set of values.

Imagine (illustratively):

0
±0.5
±1
±2
±4
±8
...

Now try storing:

1.234

It becomes:

1

or

1.5

depending on the encoding.

Try storing:

0.093

It may become:

0

or

0.125

The quantization error is much larger than FP8.

This is why plain FP4 is rarely used by itself. Instead, it is paired with block scaling, which rescales groups of values so FP4 can represent them much more accurately.

## ex val :3.14159265

| Format |               Stored Value (illustrative) |
| ------ | ----------------------------------------: |
| FP32   |                                 3.1415927 |
| FP16   |                                  3.140625 |
| BF16   | 3.140625 (similar range, lower precision) |
| FP8    |                                     3.125 |
| FP4    |                                3.0 or 4.0 |


As engineers, the key trade-off is always:

More bits → More memory, more bandwidth, higher precision.
Fewer bits → Less memory, faster computation, but more quantization error.